# Vector Databases & Embeddings - Hands-On Workshop

**Duration:** 90-120 minutes  
**Topics:** Semantic search, FAISS, Chroma, embedding generation

---

## Learning Objectives

By the end of this workshop, you will:
1. Generate embeddings using sentence-transformers
2. Build similarity search systems with FAISS
3. Create vector databases with Chroma
4. Compare semantic search vs keyword search
5. Implement production-ready document search

---

## Setup & Installation

Run this cell to install required packages:

In [1]:
# Install packages
!pip install sentence-transformers faiss-cpu chromadb scikit-learn numpy pandas -q

print("✓ Installation complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

In [2]:
# Import libraries
from sentence_transformers import SentenceTransformer
import faiss
import chromadb
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


---

## Part 1: Understanding Embeddings (20 minutes)

### What are embeddings?

Embeddings convert text into numerical vectors that capture semantic meaning.

In [3]:
# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Model loaded: {model}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)
Embedding dimension: 384


### Exercise 1.1: Generate Your First Embeddings

In [4]:
# Generate embeddings for sample sentences
sentences = [
    "The cat sat on the mat",
    "The dog played in the garden",
    "Machine learning is fascinating",
    "AI and ML are transforming technology"
]

embeddings = model.encode(sentences)

print(f"Generated {len(embeddings)} embeddings")
print(f"Each embedding has {len(embeddings[0])} dimensions")
print(f"\nFirst embedding (first 10 values):")
print(embeddings[0][:10])

Generated 4 embeddings
Each embedding has 384 dimensions

First embedding (first 10 values):
[ 0.13040186 -0.0118701  -0.02811702  0.05123865 -0.05597445  0.03019158
  0.03016127  0.02469839 -0.01837058  0.05876677]


### Exercise 1.2: Calculate Similarity

Similar sentences should have similar embeddings!

In [5]:
# Calculate cosine similarity
similarities = cosine_similarity(embeddings)

print("Similarity Matrix:")
print("="*60)
for i, sent1 in enumerate(sentences):
    print(f"\n{sent1}:")
    for j, sent2 in enumerate(sentences):
        if i != j:
            print(f"  vs '{sent2}': {similarities[i][j]:.3f}")

print("\n" + "="*60)
print("Notice: Sentences 0 & 1 (animals) are similar (0.6+)")
print("        Sentences 2 & 3 (ML/AI) are very similar (0.8+)")

Similarity Matrix:

The cat sat on the mat:
  vs 'The dog played in the garden': 0.180
  vs 'Machine learning is fascinating': -0.032
  vs 'AI and ML are transforming technology': -0.014

The dog played in the garden:
  vs 'The cat sat on the mat': 0.180
  vs 'Machine learning is fascinating': -0.027
  vs 'AI and ML are transforming technology': -0.046

Machine learning is fascinating:
  vs 'The cat sat on the mat': -0.032
  vs 'The dog played in the garden': -0.027
  vs 'AI and ML are transforming technology': 0.493

AI and ML are transforming technology:
  vs 'The cat sat on the mat': -0.014
  vs 'The dog played in the garden': -0.046
  vs 'Machine learning is fascinating': 0.493

Notice: Sentences 0 & 1 (animals) are similar (0.6+)
        Sentences 2 & 3 (ML/AI) are very similar (0.8+)


### 🎯 Practice Challenge 1.3

Find which of these queries is most similar to "I need to reset my password":
- "How do I change my login credentials?"
- "What's the weather today?"
- "Tell me about machine learning"

In [ ]:
# YOUR CODE HERE
query = "I need to reset my password"
candidates = [
    "How do I change my login credentials?",
    "What's the weather today?",
    "Tell me about machine learning"
]

# Generate embeddings
query_emb = model.encode([query])
candidate_embs = model.encode(candidates)

# Calculate similarities
sims = cosine_similarity(query_emb, candidate_embs)[0]

# Find most similar
best_idx = np.argmax(sims)
print(f"Query: '{query}'")
print(f"\nMost similar: '{candidates[best_idx]}'")
print(f"Similarity: {sims[best_idx]:.3f}")
print(f"\nAll similarities:")
for cand, sim in zip(candidates, sims):
    print(f"  {sim:.3f} - {cand}")

---

## Part 2: FAISS - Fast Similarity Search (30 minutes)

FAISS (Facebook AI Similarity Search) is optimized for searching millions of vectors.

### Exercise 2.1: Build Your First FAISS Index

In [6]:
# Sample FAQ dataset
faqs = [
    "How do I reset my password?",
    "What are your business hours?",
    "How do I track my order?",
    "What is your return policy?",
    "How do I contact customer support?",
    "Can I change my shipping address?",
    "How do I cancel my order?",
    "Do you ship internationally?",
    "How long does delivery take?",
    "What payment methods do you accept?"
]

# Generate embeddings
faq_embeddings = model.encode(faqs).astype('float32')

print(f"Created embeddings for {len(faqs)} FAQs")
print(f"Embedding shape: {faq_embeddings.shape}")

Created embeddings for 10 FAQs
Embedding shape: (10, 384)


In [7]:
# Create FAISS index
dimension = faq_embeddings.shape[1]  # 384

# Normalize for cosine similarity
faiss.normalize_L2(faq_embeddings)

# Build index (IndexFlatIP for cosine similarity)
index = faiss.IndexFlatIP(dimension)
index.add(faq_embeddings)

print(f"FAISS index built!")
print(f"Total vectors in index: {index.ntotal}")
print(f"Index is trained: {index.is_trained}")

FAISS index built!
Total vectors in index: 10
Index is trained: True


### Exercise 2.2: Search the Index

In [9]:
# User query
user_query = "I can't log into my account"

# Generate query embedding
query_embedding = model.encode([user_query]).astype('float32')
faiss.normalize_L2(query_embedding)

# Search for top 3 similar FAQs
k = 2
similarities, indices = index.search(query_embedding, k)

print(f"User query: '{user_query}'")
print(f"\nTop {k} similar FAQs:")
print("="*60)
for i, (idx, sim) in enumerate(zip(indices[0], similarities[0]), 1):
    print(f"{i}. {faqs[idx]}")
    print(f"   Similarity: {sim:.3f}")
    print()

User query: 'I can't log into my account'

Top 2 similar FAQs:
1. How do I reset my password?
   Similarity: 0.579

2. How do I contact customer support?
   Similarity: 0.326



### 🎯 Practice Challenge 2.3

Try these queries and see what the system finds:
1. "Where is my package?"
2. "I want to send something back"
3. "How can I reach you?"

In [10]:
# YOUR CODE HERE
test_queries = [
    "Where is my package?",
    "I want to send something back",
    "How can I reach you?"
]

for query in test_queries:
    query_emb = model.encode([query]).astype('float32')
    faiss.normalize_L2(query_emb)
    similarities, indices = index.search(query_emb, 2)

    print(f"Query: '{query}'")
    print(f"Top match: {faqs[indices[0][0]]} (similarity: {similarities[0][0]:.3f})")
    print()

Query: 'Where is my package?'
Top match: How do I track my order? (similarity: 0.375)

Query: 'I want to send something back'
Top match: How do I cancel my order? (similarity: 0.356)

Query: 'How can I reach you?'
Top match: How do I contact customer support? (similarity: 0.287)



### Exercise 2.4: Performance with Larger Dataset

In [11]:
import time

# Create larger dataset (1000 documents)
large_docs = [f"Document {i}: This is about topic {i % 10}" for i in range(1000)]
large_embeddings = model.encode(large_docs, show_progress_bar=True).astype('float32')
faiss.normalize_L2(large_embeddings)

# Build index
large_index = faiss.IndexFlatIP(dimension)
large_index.add(large_embeddings)

# Test search speed
query_emb = model.encode(["topic 5"]).astype('float32')
faiss.normalize_L2(query_emb)

start = time.time()
similarities, indices = large_index.search(query_emb, 5)
search_time = time.time() - start

print(f"Searched 1,000 documents in {search_time*1000:.2f}ms")
print(f"\nTop 3 results:")
for i in range(3):
    print(f"{i+1}. {large_docs[indices[0][i]]} (similarity: {similarities[0][i]:.3f})")

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Searched 1,000 documents in 0.38ms

Top 3 results:
1. Document 5: This is about topic 5 (similarity: 0.751)
2. Document 205: This is about topic 5 (similarity: 0.725)
3. Document 235: This is about topic 5 (similarity: 0.724)


---

## Part 3: Chroma - Easy Vector Database (25 minutes)

Chroma makes it simple: automatic embeddings, persistence, and metadata filtering!

### Exercise 3.1: Create Your First Collection

In [ ]:
# Initialize Chroma client
client = chromadb.Client()

# Create collection
collection = client.create_collection(name="tech_docs")

print(f"Collection created: {collection.name}")

### Exercise 3.2: Add Documents (Chroma generates embeddings automatically!)

In [ ]:
# Add documents with metadata
documents = [
    "Python is a high-level programming language",
    "JavaScript is used for web development",
    "Machine learning is a subset of AI",
    "Deep learning uses neural networks",
    "Data science involves statistics and programming"
]

metadatas = [
    {"category": "programming", "language": "python"},
    {"category": "programming", "language": "javascript"},
    {"category": "ai", "subcategory": "ml"},
    {"category": "ai", "subcategory": "dl"},
    {"category": "data_science", "skills": "statistics"}
]

ids = [f"doc{i}" for i in range(len(documents))]

# Add to collection (Chroma handles embeddings!)
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Added {collection.count()} documents")

### Exercise 3.3: Query the Collection

In [ ]:
# Simple query
results = collection.query(
    query_texts=["artificial intelligence"],
    n_results=3
)

print("Query: 'artificial intelligence'")
print("\nTop 3 results:")
print("="*60)
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
), 1):
    print(f"{i}. {doc}")
    print(f"   Metadata: {meta}")
    print(f"   Distance: {dist:.3f}")
    print()

### Exercise 3.4: Metadata Filtering

In [ ]:
# Query with metadata filter
results = collection.query(
    query_texts=["coding tutorial"],
    n_results=3,
    where={"category": "programming"}  # Only programming docs
)

print("Query: 'coding tutorial'")
print("Filter: category = 'programming'")
print("\nResults:")
print("="*60)
for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(f"• {doc}")
    print(f"  Category: {meta['category']}")
    print()

### 🎯 Practice Challenge 3.5

Add 3 more documents about databases, then query for "data storage" filtering by category="database"

In [ ]:
# YOUR CODE HERE
new_docs = [
    "SQL is used for relational databases",
    "MongoDB is a NoSQL database",
    "PostgreSQL is an open-source database"
]

new_metas = [
    {"category": "database", "type": "sql"},
    {"category": "database", "type": "nosql"},
    {"category": "database", "type": "sql"}
]

new_ids = [f"doc{i+5}" for i in range(len(new_docs))]

collection.add(documents=new_docs, metadatas=new_metas, ids=new_ids)

# Query
results = collection.query(
    query_texts=["data storage"],
    n_results=3,
    where={"category": "database"}
)

print("Results for 'data storage' in database category:")
for doc in results['documents'][0]:
    print(f"• {doc}")

---

## Part 4: Semantic Search vs Keyword Search (15 minutes)

Let's compare traditional keyword search with semantic search!

In [ ]:
# Product catalog
products = [
    "MacBook Pro 16-inch laptop",
    "Dell XPS 15 notebook computer",
    "HP Pavilion portable workstation",
    "iPad Pro 12.9-inch tablet",
    "Wireless mouse and keyboard combo",
    "USB-C charging cable",
    "Laptop stand for desk",
    "Gaming desktop PC"
]

query = "laptop computer"

In [ ]:
# KEYWORD SEARCH
print("KEYWORD SEARCH")
print("="*60)
print(f"Query: '{query}'\n")

keyword_results = [p for p in products if "laptop" in p.lower() or "computer" in p.lower()]

print("Results:")
for p in keyword_results:
    print(f"✓ {p}")

print(f"\nFound: {len(keyword_results)} products")
print("\nMissed:")
missed = [p for p in products if p not in keyword_results]
for p in missed:
    if any(word in p.lower() for word in ['macbook', 'dell', 'hp', 'pavilion']):
        print(f"✗ {p} (IS a laptop but no keywords!)")

In [ ]:
# SEMANTIC SEARCH
print("\nSEMANTIC SEARCH")
print("="*60)
print(f"Query: '{query}'\n")

# Generate embeddings
product_embeddings = model.encode(products)
query_emb = model.encode([query])

# Calculate similarities
similarities = cosine_similarity(query_emb, product_embeddings)[0]

# Sort by similarity
ranked = sorted(zip(products, similarities), key=lambda x: x[1], reverse=True)

print("Results (top 5 by similarity):")
for product, sim in ranked[:5]:
    print(f"{sim:.3f} - {product}")

print("\n✓ Found MacBook, Dell XPS, HP Pavilion (all laptops!)")
print("✓ Even though they don't contain 'laptop computer' keywords")

---

## Part 5: Final Project - Complete Search System (30 minutes)

Build a production-ready document search system!

In [ ]:
class DocumentSearchEngine:
    """
    Production-ready document search with FAISS
    """
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.dimension = self.model.get_sentence_embedding_dimension()
        self.index = None
        self.documents = []
        self.metadatas = []

    def add_documents(self, documents, metadatas=None):
        """Add documents to the search index"""
        print(f"Adding {len(documents)} documents...")

        # Generate embeddings
        embeddings = self.model.encode(
            documents,
            batch_size=32,
            show_progress_bar=True
        ).astype('float32')

        # Normalize for cosine similarity
        faiss.normalize_L2(embeddings)

        # Create or update index
        if self.index is None:
            self.index = faiss.IndexFlatIP(self.dimension)

        self.index.add(embeddings)
        self.documents.extend(documents)

        if metadatas:
            self.metadatas.extend(metadatas)
        else:
            self.metadatas.extend([{}] * len(documents))

        print(f"✓ Total documents: {len(self.documents)}")

    def search(self, query, top_k=5, filter_fn=None):
        """Search for similar documents"""
        # Generate query embedding
        query_emb = self.model.encode([query]).astype('float32')
        faiss.normalize_L2(query_emb)

        # Search (get extra results for filtering)
        search_k = min(top_k * 10, self.index.ntotal)
        similarities, indices = self.index.search(query_emb, search_k)

        # Format and filter results
        results = []
        for sim, idx in zip(similarities[0], indices[0]):
            result = {
                'document': self.documents[idx],
                'metadata': self.metadatas[idx],
                'similarity': float(sim)
            }

            # Apply custom filter if provided
            if filter_fn is None or filter_fn(result):
                results.append(result)

            if len(results) >= top_k:
                break

        return results

    def stats(self):
        """Get search engine statistics"""
        return {
            'total_documents': len(self.documents),
            'embedding_dimension': self.dimension,
            'index_size': self.index.ntotal if self.index else 0
        }

In [ ]:
# Create search engine
engine = DocumentSearchEngine()

# Sample knowledge base
kb_docs = [
    "To reset your password, click 'Forgot Password' on the login page",
    "Our business hours are Monday-Friday, 9 AM to 5 PM EST",
    "Track your order by visiting the 'Order History' page in your account",
    "We accept returns within 30 days with original receipt",
    "Contact customer support at support@company.com or call 1-800-123-4567",
    "Standard shipping takes 3-5 business days",
    "Yes, we ship internationally to over 50 countries",
    "You can cancel your order within 24 hours of purchase",
    "We accept Visa, Mastercard, American Express, and PayPal",
    "Create an account to save your preferences and order history"
]

kb_metadata = [
    {"category": "account", "topic": "password"},
    {"category": "general", "topic": "hours"},
    {"category": "orders", "topic": "tracking"},
    {"category": "returns", "topic": "policy"},
    {"category": "support", "topic": "contact"},
    {"category": "shipping", "topic": "delivery"},
    {"category": "shipping", "topic": "international"},
    {"category": "orders", "topic": "cancellation"},
    {"category": "payment", "topic": "methods"},
    {"category": "account", "topic": "registration"}
]

# Add documents
engine.add_documents(kb_docs, kb_metadata)

# Show stats
print(f"\nEngine stats: {engine.stats()}")

In [ ]:
# Test searches
test_queries = [
    "I can't log into my account",
    "Where is my package?",
    "I want to return a product",
    "How can I pay?"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("="*60)

    results = engine.search(query, top_k=2)

    for i, result in enumerate(results, 1):
        print(f"{i}. {result['document']}")
        print(f"   Category: {result['metadata']['category']}")
        print(f"   Similarity: {result['similarity']:.3f}")
        print()

### 🎯 Final Challenge

Add category filtering to your searches!

In [ ]:
# Search with category filter
query = "I need help"
category = "account"

results = engine.search(
    query,
    top_k=3,
    filter_fn=lambda r: r['metadata'].get('category') == category
)

print(f"Query: '{query}'")
print(f"Filter: category='{category}'")
print("\nResults:")
for result in results:
    print(f"• {result['document']}")
    print(f"  Topic: {result['metadata']['topic']}")

---

## Summary & Key Takeaways

### What You Learned

1. **Embeddings** convert text to vectors that capture meaning
2. **FAISS** provides fast similarity search for millions of vectors
3. **Chroma** makes it easy with automatic embeddings and persistence
4. **Semantic search** understands meaning, not just keywords

### Best Practices

- ✅ Use sentence-transformers for free, local embeddings
- ✅ Normalize embeddings before using FAISS IndexFlatIP
- ✅ Use Chroma for quick projects, FAISS for scale
- ✅ Always filter results by similarity threshold

### Next Steps

- Try with your own documents
- Experiment with different embedding models
- Build a chatbot using semantic search
- Implement RAG (Retrieval Augmented Generation)

---

**🎉 Congratulations! You've built a production-ready semantic search system!**